# Pipeline: Qwen2.5-32B-Instruct

This notebook is set up for a Colab workflow: install dependencies, confirm the GPU, pull the repo, mount Drive, configure dataset and output paths, then run activation extraction and verify the saved HDF5 file.

## 1. Runtime Setup

Run the next two cells first. They install the Python dependencies used by the extraction pipeline and show the current GPU.

In [1]:
!pip install -q transformers accelerate h5py huggingface_hub scikit-learn

In [2]:
!nvidia-smi

Thu Apr 30 18:49:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Repo and Drive

The next cells pull the latest repo into `/content` and mount Google Drive. The code lives best on local Colab disk for speed, while datasets and final outputs can stay in Drive for persistence.

In [3]:
import os, sys

REPO_DIR = '/content/emotion-mechanisms-llm'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/daspushpita/emotion-mechanisms-llm.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

for p in [f'{REPO_DIR}/src', f'{REPO_DIR}/scripts']:
    if p not in sys.path:
        sys.path.insert(0, p)

Cloning into '/content/emotion-mechanisms-llm'...
remote: Enumerating objects: 273, done.
remote: Counting objects: 100% (273/273), done.
remote: Compressing objects: 100% (165/165), done.
remote: Total 273 (delta 154), reused 208 (delta 90), pack-reused 0 (from 0)
Receiving objects: 100% (273/273), 1.50 MiB | 18.33 MiB/s, done.
Resolving deltas: 100% (154/154), done.


In [4]:
from google.colab import drive
drive.mount('/content/drive')

from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv("/content/drive/MyDrive/.secrets/hf.env")
hf_token = os.getenv("HF_TOKEN")
assert hf_token is not None, "HF_TOKEN not found"
login(token=hf_token)

Mounted at /content/drive


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## 3. Configure Paths and Model

Edit `DATA_ROOT` if your Drive folder name differs. This is also the place to switch between smoke-test settings and a real 32B run. If you keep the sample values below, note that the current config still points to a 7B output filename and a 7B analysis model.

In [5]:
# Patch config BEFORE importing build_emotion_vectors so its module-level
# `from emotion_mechanisms.config import ...` picks up the Drive paths.
import emotion_mechanisms.config as cfg
from pathlib import Path

REPO_DIR = Path("/content/emotion-mechanisms-llm")
DATA_ROOT = Path("/content/drive/MyDrive/emotion-mechanisms-llm")

LAYER_INDICES_32B = list(range(64))

cfg.EMOTIONAL_STORIES_DATASET = DATA_ROOT / "datasets/processed/emotional_stories_qwen32B_v1_clean.jsonl"
cfg.NEUTRAL_STORIES_DATASET   = DATA_ROOT / "datasets/processed/neutral_stories_qwen32B_v1.jsonl"
cfg.ACTIVATIONS_PATH          = Path("/content/activations_32b.h5")
cfg.ANALYSIS_MODEL_32B        = "Qwen/Qwen2.5-32B-Instruct"
cfg.TOKEN_POSITION            = "mean"
cfg.LAYER_INDICES_32B = LAYER_INDICES_32B


assert cfg.EMOTIONAL_STORIES_DATASET.exists(), f'Missing: {cfg.EMOTIONAL_STORIES_DATASET}'
assert cfg.NEUTRAL_STORIES_DATASET.exists(),   f'Missing: {cfg.NEUTRAL_STORIES_DATASET}'
cfg.ACTIVATIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
print('Datasets found. Output ->', cfg.ACTIVATIONS_PATH)

Datasets found. Output -> /content/activations_32b.h5


In [6]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


## 4. Authenticate and Run

Hugging Face login is only needed if the model download requires authentication or if you want authenticated rate limits. The extraction cell below uses `max_stories=1` as a quick smoke test; increase it once the pipeline behaves the way you want.

In [7]:
import importlib
import build_emotion_vectors
importlib.reload(build_emotion_vectors)

build_emotion_vectors.main()

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Emotional stories:   0%|          | 0/2295 [00:00<?, ?it/s]

Neutral stories:   0%|          | 0/48 [00:00<?, ?it/s]

In [8]:
import shutil

drive_out = DATA_ROOT / "results/activations/activations_32b.h5"
drive_out.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2("/content/activations_32b.h5", drive_out)
print("Copied to", drive_out)

Copied to /content/drive/MyDrive/emotion-mechanisms-llm/results/activations/activations_32b.h5


In [9]:
import h5py
with h5py.File(cfg.ACTIVATIONS_PATH, 'r') as f:
    print('Top-level keys:', list(f.keys()))
    print('Emotions stored:', list(f['emotional'].keys()))
    first_layer = list(f['emotional/happy'].keys())[0]
    print(f'emotional/happy/{first_layer} shape:', f[f'emotional/happy/{first_layer}'].shape)

Top-level keys: ['emotional', 'neutral']
Emotions stored: ['afraid', 'angry', 'calm', 'desperate', 'guilty', 'happy', 'inspired', 'loving', 'nervous', 'proud', 'sad', 'surprised']
emotional/happy/layer_0 shape: (190, 5120)


## 5. Verify the Saved Activations

These checks confirm that the HDF5 file was written correctly. First inspect the top-level groups and one example tensor shape, then run a quick numeric sanity check across a few layers per emotion.

In [10]:
import numpy as np
import h5py

with h5py.File(cfg.ACTIVATIONS_PATH, 'r') as f:
    print(f"{'Emotion':<12} {'Layer':<10} {'Mean':>10} {'Std':>10} {'NaN':>6} {'Zero':>6}")
    print("-" * 56)
    for emotion in f['emotional'].keys():
        layers = list(f[f'emotional/{emotion}'].keys())
        # spot-check first, middle, last layer
        for layer in [layers[0], layers[len(layers)//2], layers[-1]]:
            arr = f[f'emotional/{emotion}/{layer}'][:]
            print(f"{emotion:<12} {layer:<10} {arr.mean():>10.3f} {arr.std():>10.3f} "
                f"{str(np.isnan(arr).any()):>6} {str((arr==0).all(-1).any()):>6}")
    
    # Also check neutral
    print("\nNeutral keys:", list(f['neutral'].keys()))
    n_layers = list(f['emotional/happy'].keys())
    print(f"Layers saved per emotion: {len(n_layers)} (e.g. {n_layers[0]} … {n_layers[-1]})")

Emotion      Layer            Mean        Std    NaN   Zero
--------------------------------------------------------
afraid       layer_0        -0.004      0.392  False  False
afraid       layer_38       -0.012      4.188  False  False
afraid       layer_9        -0.011      3.610  False  False
angry        layer_0        -0.004      0.392  False  False
angry        layer_38       -0.012      4.016  False  False
angry        layer_9        -0.009      3.470  False  False
calm         layer_0        -0.004      0.391  False  False
calm         layer_38       -0.004      4.207  False  False
calm         layer_9        -0.009      3.651  False  False
desperate    layer_0        -0.004      0.391  False  False
desperate    layer_38       -0.012      4.217  False  False
desperate    layer_9        -0.010      3.630  False  False
guilty       layer_0        -0.004      0.391  False  False
guilty       layer_38       -0.011      3.848  False  False
guilty       layer_9        -0.010      3.2